# ЧЕКПОИНТ 5

In [32]:
import statsforecast

!pip install statsforecast

In [33]:
import random
import numpy as np
import pandas as pd
import os
import warnings

from statsforecast import StatsForecast
from tqdm import tqdm
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error
import time

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)


seed_everything()

In [34]:
!pip install catboost

In [35]:
import lightgbm as lgb
import catboost as cb
import xgboost as xgb

In [36]:
df = pd.read_csv('data/moex/moex_top10_liquid.csv')
df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'])
df = df.sort_values(['SECID', 'TRADEDATE']).reset_index(drop=True)

def load_macro_safe(path, sep=',', decimal='.'):
    m = pd.read_csv(path, sep=sep, decimal=decimal)

    date_col, val_col = m.columns[0], m.columns[1]
    m[date_col] = pd.to_datetime(m[date_col])
    return m[[date_col, val_col]].rename(columns={date_col: 'TRADEDATE', val_col: 'VALUE'})

brent = load_macro_safe('data/features/brent_price.csv').rename(columns={'VALUE': 'BRENT'})
usd_rub = load_macro_safe('data/features/usd_rub.csv').rename(columns={'VALUE': 'USD_RUB'})
key_rate = load_macro_safe('data/features/key_rate.csv', sep=';', decimal=',').rename(columns={'VALUE': 'KEY_RATE'})

# по дате
for macro_df in [brent, usd_rub, key_rate]:
    df = df.merge(macro_df, on='TRADEDATE', how='left')

for col in ['BRENT', 'USD_RUB', 'KEY_RATE']:
    df[col] = df.groupby('SECID')[col].ffill()

print(df[['TRADEDATE', 'SECID', 'BRENT', 'USD_RUB', 'KEY_RATE']].head())

   TRADEDATE SECID   BRENT  USD_RUB  KEY_RATE
0 2014-06-09  GAZP  110.55      NaN       7.5
1 2014-06-10  GAZP  109.18  34.3303       7.5
2 2014-06-11  GAZP  109.83  34.3681       7.5
3 2014-06-12  GAZP  112.18  34.3227       7.5
4 2014-06-13  GAZP  113.15  34.3227       7.5


Добавим переменные

### Созданные признаки:

#### 1. Технические индикаторы:
- `LOG_RET` - логарифмическая доходность (целевая переменная)
- `vol_20` - волатильность за 20 дней (скользящее стандартное отклонение)
- `ma_5_ratio` - отношение цены к скользящей средней за 5 дней
- `ret_lag1` - лаг доходности (t-1)

#### 2. Макроэкономические признаки с лагами и скользящими средними:
- `BRENT_lag3`, `BRENT_ma30`, `BRENT_ma180` - цена нефти с разными горизонтами
- `USD_RUB_lag3`, `USD_RUB_ma30`, `USD_RUB_ma365` - курс валюты
- `KEY_RATE_lag3`, `KEY_RATE_ma30`, `KEY_RATE_ma180` - ключевая ставка

In [37]:
HORIZON = 30

for secid in tqdm(df['SECID'].unique()):
    mask = df['SECID'] == secid

    # Технические
    df.loc[mask, 'LOG_RET'] = np.log(df.loc[mask, 'LEGALCLOSEPRICE'] / df.loc[mask, 'LEGALCLOSEPRICE'].shift(1))
    df.loc[mask, 'vol_20'] = df.loc[mask, 'LOG_RET'].rolling(20).std()
    df.loc[mask, 'ma_5_ratio'] = df.loc[mask, 'LEGALCLOSEPRICE'] / df.loc[mask, 'LEGALCLOSEPRICE'].rolling(5).mean()
    df.loc[mask, 'ret_lag1'] = df.loc[mask, 'LOG_RET'].shift(1)

    # окна (3 дня, месяц, полгода, год)
    for col in ['BRENT', 'USD_RUB', 'KEY_RATE']:
        if col in df.columns:
            df.loc[mask, f'{col}_lag3'] = df.loc[mask, col].shift(3)
            df.loc[mask, f'{col}_ma30'] = df.loc[mask, col].rolling(30).mean()
            df.loc[mask, f'{col}_ma180'] = df.loc[mask, col].rolling(180).mean()
            df.loc[mask, f'{col}_ma365'] = df.loc[mask, col].rolling(365).mean()

100%|██████████| 10/10 [00:00<00:00, 121.49it/s]


In [38]:
# Кластеризация активов
def calc_group_stats(g):
    """Считаем статистику по каждому тикеру для кластеризации"""
    return pd.Series({
        'vol_mean': g['vol_20'].mean(),
        'ret_mean': g['ret_lag1'].mean(),
        'brent_corr': g['BRENT'].corr(g['LOG_RET'])
    })

# Группируем
cluster_stats = df.groupby('SECID').apply(calc_group_stats).dropna()

# Масштабируем для кластеризации
scaler = StandardScaler()
X_c = scaler.fit_transform(cluster_stats[['vol_mean', 'ret_mean', 'brent_corr']])

km = KMeans(n_clusters=3, random_state=42, n_init='auto')
cluster_stats['CLUSTER'] = km.fit_predict(X_c)

# Присоединяем кластер обратно к основному датасету
df = df.merge(cluster_stats[['CLUSTER']], left_on='SECID', right_index=True, how='left')

df = df.dropna(subset=['LOG_RET', 'ret_lag1', 'vol_20']).reset_index(drop=True)

In [39]:
def calc_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def calc_mase(y_true, y_pred, y_train):
    mae_model = np.abs(y_true - y_pred).mean()
    naive = np.abs(y_train.diff().dropna())
    mae_naive = naive.mean()
    return mae_model / mae_naive if mae_naive > 0 else np.nan


Используем нелинейные модели

Обучили LightGBM и CatBoost на трёх горизонтах (10, 20, 30 дней). Бустинги работают быстро и ловят нелинейные зависимости, которые линейные модели пропускают. Для сравнения есть SARIMA — она из прошлого чекпойнта, без экзогенных признаков

In [40]:
FEATURES = [
    'ret_lag1', 'vol_20', 'ma_5_ratio',
    'BRENT_lag3', 'BRENT_ma30', 'BRENT_ma180',
    'USD_RUB_lag3', 'USD_RUB_ma30', 'USD_RUB_ma365',
    'KEY_RATE_lag3', 'KEY_RATE_ma30', 'KEY_RATE_ma180',
]
results = []

# Тестируем разные горизонты
for H in [10, 20, 30]:
    print(f"\nHorizon {H}")
    for secid in tqdm(df['SECID'].unique()):
        d = df[df['SECID']==secid].copy()
        if len(d) < 1000: continue

        train = d.iloc[:-H]
        test = d.iloc[-H:]

        X_tr, y_tr = train[FEATURES], train['LOG_RET']
        X_te, y_te = test[FEATURES], test['LOG_RET']

        # LightGBM
        t0 = time.time()
        mdl = lgb.LGBMRegressor(n_estimators=150, max_depth=4, learning_rate=0.05, verbose=-1)
        mdl.fit(X_tr, y_tr)
        preds = mdl.predict(X_te)
        t_lgb = time.time() - t0
        results.append({'SECID':secid, 'HORIZON':H, 'MODEL':'LightGBM',
                       'RMSE':calc_rmse(y_te, preds), 'MASE':calc_mase(y_te, preds, y_tr), 'TIME':round(t_lgb,2)})

        # CatBoost
        t0 = time.time()
        mdl = cb.CatBoostRegressor(iterations=150, depth=4, learning_rate=0.05, verbose=0, allow_writing_files=False)
        mdl.fit(X_tr, y_tr)
        preds = mdl.predict(X_te)
        t_cb = time.time() - t0
        results.append({'SECID':secid, 'HORIZON':H, 'MODEL':'CatBoost',
                       'RMSE':calc_rmse(y_te, preds), 'MASE':calc_mase(y_te, preds, y_tr), 'TIME':round(t_cb,2)})


Horizon 10


100%|██████████| 10/10 [00:01<00:00,  7.62it/s]



Horizon 20


100%|██████████| 10/10 [00:01<00:00,  7.53it/s]



Horizon 30


100%|██████████| 10/10 [00:01<00:00,  7.40it/s]


Сводная таблица

In [41]:
df_all = pd.DataFrame(results)
df_final = pd.concat([df_all], ignore_index=True)

summary = df_final.groupby('MODEL').agg({
    'RMSE':'mean', 'MASE':'mean', 'TIME':'mean'
}).round(4).sort_values('RMSE')
display(summary)

df_final.to_csv('ml_experiments.csv', index=False)

,RMSE,MASE,TIME
MODEL,,,
LightGBM,0.0080,0.4205,0.0286
CatBoost,0.0086,0.4501,0.1548


SARIMA

RMSE: 0.008872418574827115

MASE: 0.4163219980057422

LightGBM показал лучший RMSE (0.0080), а SARIMA — чуть лучший MASE (0.4163). Признаки немного улучшили результат, но не радикально

In [42]:
!pip install autots

In [43]:
from autots import AutoTS
import warnings
import time
import numpy as np
import pandas as pd
from tqdm import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)

if 'log_returns' not in df.columns:
    print("Создаем колонку log_returns...")
    df['log_returns'] = df.groupby('SECID')['LEGALCLOSEPRICE'].transform(
        lambda x: np.log(x / x.shift(1))
    )

autots_res = []
df_fe = df

# Используем df_fe (с признаками) вместо df
for secid in tqdm(df_fe['SECID'].unique()[:3], desc="AutoTS (топ-3 для экономии времени)"):
    d = df_fe[df_fe['SECID'] == secid].copy()
    d = d.sort_values('TRADEDATE').reset_index(drop=True)

    # Проверяем наличие log_returns
    if 'log_returns' not in d.columns:
        d['log_returns'] = np.log(d['LEGALCLOSEPRICE'] / d['LEGALCLOSEPRICE'].shift(1))

    d = d.dropna(subset=['log_returns']).reset_index(drop=True)

    train = d.iloc[:-30].copy()
    test = d.iloc[-30:].copy()

    # Формат для AutoTS: строго ['Date', 'Value']
    ts_train = train[['TRADEDATE', 'log_returns']].rename(
        columns={'TRADEDATE': 'Date', 'log_returns': 'Value'}
    )

    # Подготовка экзогенных признаков
    exog_cols = []
    for col in ['brent_price_ma_7', 'usdrub_rate_ma_7', 'key_rate_ma_7',
                'brent_price', 'usdrub_rate', 'key_rate']:
        if col in d.columns:
            exog_cols.append(col)

    if len(exog_cols) == 0:
        exog_cols = ['brent_price', 'usdrub_rate', 'key_rate']

    print(f"  {secid}: используем экзогенные признаки: {exog_cols[:3]}")

    exog_tr = train[exog_cols].fillna(method='ffill').fillna(method='bfill')
    exog_te = test[exog_cols].fillna(method='ffill').fillna(method='bfill')

    t0 = time.time()

    try:
        model = AutoTS(
            forecast_length=30,
            frequency='D',
            ensemble='simple',
            max_generations=2,
            num_validations=2,
            model_list=["LastValueNaive", "AverageValueNaive", "ETS"],
            drop_most_recent=1,
            n_jobs=1
        )

        model.fit(ts_train, exogenous_regressor_matrix=exog_tr)
        forecast = model.predict(exogenous_regressor_matrix=exog_te)
        preds = forecast.forecast['Value'].values
        t_auto = time.time() - t0

        # Получаем имя лучшей модели
        if hasattr(model, 'best_model_name'):
            best_model_name = model.best_model_name
        elif hasattr(model, 'best_model'):
            best_model_name = str(model.best_model)
        else:
            best_model_name = 'Unknown'

        # Рассчитываем метрики
        y_test = test['log_returns'].values
        y_train = train['log_returns'].values

        rmse_val = np.sqrt(np.mean((y_test - preds) ** 2))

        # MASE
        naive_errors = np.abs(np.diff(y_train))
        mae_naive = naive_errors.mean() if len(naive_errors) > 0 else 1
        mae_model = np.abs(y_test - preds).mean()
        mase_val = mae_model / mae_naive if mae_naive != 0 else np.nan

        autots_res.append({
            'SECID': secid,
            'HORIZON': 30,
            'MODEL': 'AutoTS',
            'RMSE': rmse_val,
            'MASE': mase_val,
            'TIME': round(t_auto, 2),
            'BestModel': best_model_name
        })
        print(f"  ✓ {secid}: RMSE={rmse_val:.6f}, Best={best_model_name}")

    except Exception as e:
        print(f"  ✗ {secid}: Ошибка - {str(e)[:100]}")

        # Пробуем без экзогенных признаков
        try:
            print(f"  ↻ {secid}: пробуем без экзогенных признаков...")
            model = AutoTS(
                forecast_length=30,
                frequency='D',
                ensemble='simple',
                max_generations=2,
                num_validations=2,
                model_list=["LastValueNaive", "AverageValueNaive", "ETS"],
                n_jobs=1
            )
            model.fit(ts_train)
            forecast = model.predict()
            preds = forecast.forecast['Value'].values

            y_test = test['log_returns'].values
            y_train = train['log_returns'].values

            rmse_val = np.sqrt(np.mean((y_test - preds) ** 2))
            naive_errors = np.abs(np.diff(y_train))
            mae_naive = naive_errors.mean() if len(naive_errors) > 0 else 1
            mae_model = np.abs(y_test - preds).mean()
            mase_val = mae_model / mae_naive if mae_naive != 0 else np.nan

            autots_res.append({
                'SECID': secid,
                'HORIZON': 30,
                'MODEL': 'AutoTS (no exog)',
                'RMSE': rmse_val,
                'MASE': mase_val,
                'TIME': round(time.time() - t0, 2),
                'BestModel': 'LastValueNaive'
            })
            print(f"  ✓ {secid} (без экзогенных): RMSE={rmse_val:.6f}")

        except Exception as e2:
            autots_res.append({
                'SECID': secid,
                'HORIZON': 30,
                'MODEL': 'AutoTS',
                'RMSE': np.nan,
                'MASE': np.nan,
                'TIME': time.time() - t0,
                'BestModel': f'Error'
            })

df_autots = pd.DataFrame(autots_res)
print(df_autots)

Создаем колонку log_returns...


AutoTS (топ-3 для экономии времени):   0%|          | 0/3 [00:00<?, ?it/s]

  GAZP: используем экзогенные признаки: ['brent_price', 'usdrub_rate', 'key_rate']


KeyError: "None of [Index(['brent_price', 'usdrub_rate', 'key_rate'], dtype='object')] are in the [columns]"